# testing mean calculation from reshapr file and from direct model output to see where i'm going wrong

In [1]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
# bathymetry for masking land values
bath = xr.open_dataset("/home/rbeutel/MEOPAR/grid/bathymetry_202108.nc") 

In [52]:
# for masking the different regions of analysis
def bathmask(mydata,region):

    # region can be jdf, pug (puget sound), nsg (northern SoG), and csg (central SoG)
    # based on boxes on map
    if region == 'jdf':
        out_bath = (mydata.Bathymetry[310:360,50:100]>0).rename({'y': 'gridY','x': 'gridX'})
    
    elif region == 'pug':
        out_bath = (mydata.Bathymetry[80:130,220:270]>0).rename({'y': 'gridY','x': 'gridX'})

    elif region == 'nsg':
        out_bath = (mydata.Bathymetry[650:700,130:180]>0).rename({'y': 'gridY','x': 'gridX'})

    elif region == 'csg':
        out_bath = (mydata.Bathymetry[460:510,240:290]>0).rename({'y': 'gridY','x': 'gridX'})

    else:
        print("invalid region name")

    return out_bath


# booleans describing the different regions of analysis
def booleans(mydata,region,depth):

    if depth==50:
        depth_bool = mydata.deptht == mydata.deptht[23]
    elif depth==0:
        depth_bool = mydata.deptht == mydata.deptht[0]
    elif depth==300:
        depth_bool = mydata.deptht == mydata.deptht[35]


    # region can be jdf, pug (puget sound), nsg (northern SoG), and csg (central SoG)
    # based on boxes on map
    if region == 'jdf':
        y_bool = (mydata.y >=310) & (mydata.y <360) 
        x_bool = (mydata.x >=50) & (mydata.x<100)
    
    elif region == 'pug':
        y_bool = (mydata.y >=80) & (mydata.y <130)
        x_bool = (mydata.x >=220) & (mydata.x<270)

    elif region == 'nsg':
        y_bool = (mydata.y >=650) & (mydata.y <700)
        x_bool = (mydata.x >=130) & (mydata.x<180)

    elif region == 'csg':
        y_bool = (mydata.y >=460) & (mydata.y <510)
        x_bool = (mydata.x >=240) & (mydata.x<290)

    else:
        print("invalid region name")

    return depth_bool, y_bool, x_bool

In [7]:
# original salishseacast
old = xr.open_dataset("/home/rbeutel/projects/def-allen/SalishSea/nowcast-green.202111/06jul20/SalishSea_1d_20200706_20200706_grid_T.nc")

# projection direct output example
mod = xr.open_dataset('/home/rbeutel/MEOPAR/analysis/output/ForThalweg/All_1d_20200101_20201231_pri_T_20200706-20200706.nc')

# reshapher file
# choosing Jdf bc for some reason its COLDER than old when we calculate it from the reshaper file but not when we calculate it from the original salishseacast
res = xr.open_dataset('/home/rbeutel/MEOPAR/analysis/output/All_2020spin_primarytracers_jdf50_20200101_20201231.nc')

# old calc

In [54]:
data = old.where(~np.isnan(bath.Bathymetry))

depth_bool, y_bool, x_bool = booleans(data,'jdf',depth=50)

# how i currently have it calculated
np.mean(data.votemper[:,depth_bool,y_bool,x_bool],axis=(1,2,3)).values

array([8.125046], dtype=float32)

In [55]:
np.nanmean(data.votemper[:,depth_bool,y_bool,x_bool],axis=(1,2,3))

array([8.125046], dtype=float32)

# new calc

In [56]:
data = mod.where(~np.isnan(bath.Bathymetry))

depth_bool, y_bool, x_bool = booleans(data,'jdf',depth=50)

# how i currently have it calculated
np.mean(data.votemper[:,depth_bool,y_bool,x_bool],axis=(1,2,3)).values

array([9.058129], dtype=float32)

### ok so slight warmer as expected

In [57]:
data.votemper[:,depth_bool,y_bool,x_bool]

<xarray.DataArray 'votemper' (time_counter: 1, deptht: 1, y: 50, x: 50)> Size: 10kB
array([[[[      nan,       nan,       nan, ..., 9.5240755, 9.57004  ,
          9.624335 ],
         [      nan,       nan,       nan, ..., 9.553041 , 9.592362 ,
          9.665973 ],
         [      nan,       nan,       nan, ..., 9.592944 , 9.641034 ,
          9.72519  ],
         ...,
         [9.577545 , 9.64806  , 9.73355  , ...,       nan,       nan,
                nan],
         [9.620245 , 9.702794 , 9.802633 , ...,       nan,       nan,
                nan],
         [9.692524 , 9.776112 , 9.847475 , ...,       nan,       nan,
                nan]]]], shape=(1, 1, 50, 50), dtype=float32)
Coordinates:
  * time_counter   (time_counter) datetime64[ns] 8B 2020-07-06T12:00:00
    time_centered  (time_counter) datetime64[ns] 8B ...
  * deptht         (deptht) float32 4B 44.52
    nav_lat        (y, x) float32 10kB ...
    nav_lon        (y, x) float32 10kB ...
Dimensions without coordinates: y, x
Attributes:
    standard_name:       sea_water_conservative_temperature
    long_name:           Conservative Temperature
    units:               degree_C
    online_operation:    average
    interval_operation:  40 s
    interval_write:      1 d
    cell_methods:        time: mean (interval: 40 s)
    cell_measures:       area: area

# reshapr

In [58]:
data = res.votemper[187,0,:,:].where(bathmask(bath,'jdf'))
data.values

array([[      nan,       nan,       nan, ..., 9.5240755, 9.57004  ,
        9.624335 ],
       [      nan,       nan,       nan, ..., 9.553041 , 9.592362 ,
        9.665973 ],
       [      nan,       nan,       nan, ..., 9.592944 , 9.641034 ,
        9.72519  ],
       ...,
       [9.577545 , 9.64806  , 9.73355  , ...,       nan,       nan,
              nan],
       [9.620245 , 9.702794 , 9.802633 , ...,       nan,       nan,
              nan],
       [9.692524 , 9.776112 , 9.847475 , ...,       nan,       nan,
              nan]], shape=(50, 50), dtype=float32)

In [59]:
data[0,:]

<xarray.DataArray 'votemper' (gridX: 50)> Size: 200B
array([      nan,       nan,       nan,       nan,       nan,       nan,
             nan,       nan,       nan,       nan,       nan,       nan,
             nan,       nan,       nan, 0.       , 0.       , 0.       ,
       9.129343 , 9.134878 , 9.1115055, 9.113596 , 9.150818 , 9.117073 ,
       9.118329 , 9.16029  , 9.163215 , 9.229328 , 9.240421 , 9.256493 ,
       9.252328 , 9.249578 , 9.227516 , 9.194407 , 9.186398 , 9.194609 ,
       9.193709 , 9.197166 , 9.18354  , 9.194411 , 9.225617 , 9.264469 ,
       9.278891 , 9.333259 , 9.374008 , 9.422815 , 9.457905 , 9.5240755,
       9.57004  , 9.624335 ], dtype=float32)
Coordinates:
  * gridX    (gridX) int64 400B 50 51 52 53 54 55 56 57 ... 93 94 95 96 97 98 99
    time     datetime64[ns] 8B 2020-07-06T12:00:00
    depth    float32 4B 44.52
    gridY    int64 8B 310
Attributes:
    units:          degree_C
    standard_name:  sea_water_conservative_temperature
    long_name:      Conservative Temperature

In [61]:
data.mean(dim=['gridY','gridX'],skipna=True)

<xarray.DataArray 'votemper' ()> Size: 4B
array(9.058129, dtype=float32)
Coordinates:
    time     datetime64[ns] 8B 2020-07-06T12:00:00
    depth    float32 4B 44.52
Attributes:
    units:          degree_C
    standard_name:  sea_water_conservative_temperature
    long_name:      Conservative Temperature

# current problem
ok so reshapr is working as expected so that's a win! my current problem is that how i'm masking according to the bathymetry is not considering that different cells should be masked as different depths. SO that means there are a lot of zero getting into the mean that shouldn't be there. This is now at least consistent between all the calculations, so the relative change is fun but the actual numbers are not. 